In [6]:
# Diagnostic cell — run this first to inspect the capabilities object structure
# Useful if the API changes and capability attributes need to be updated
m = models[0]
print(type(m.capabilities))
print(m.capabilities)
print(dir(m.capabilities))

<class 'anthropic.types.model_capabilities.ModelCapabilities'>
ModelCapabilities(batch=CapabilitySupport(supported=True), citations=CapabilitySupport(supported=True), code_execution=CapabilitySupport(supported=True), context_management=ContextManagementCapability(clear_thinking_20251015=CapabilitySupport(supported=True), clear_tool_uses_20250919=CapabilitySupport(supported=True), compact_20260112=CapabilitySupport(supported=True), supported=True), effort=EffortCapability(high=CapabilitySupport(supported=True), low=CapabilitySupport(supported=True), max=CapabilitySupport(supported=True), medium=CapabilitySupport(supported=True), supported=True), image_input=CapabilitySupport(supported=True), pdf_input=CapabilitySupport(supported=True), structured_outputs=CapabilitySupport(supported=True), thinking=ThinkingCapability(supported=True, types=ThinkingTypes(adaptive=CapabilitySupport(supported=True), enabled=CapabilitySupport(supported=True))))
['__abstractmethods__', '__annotate_func__', '__

In [8]:
# --- Setup ---
import anthropic, os
from datetime import datetime
import pandas as pd
from IPython.display import display

# Build the client and fetch the full list of available models
client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
models = list(client.models.list())

# --- Pricing table ---
# Prices in USD per million tokens (MTok), verified April 2026
# batch input = 50% of standard input price
# cache_read  = 10% of standard input price
PRICING = {
    # ── Current generation ────────────────────────────────────────
    "claude-opus-4-6":              {"input":  5.00, "output": 25.00, "cache_read": 0.50},
    "claude-sonnet-4-6":            {"input":  3.00, "output": 15.00, "cache_read": 0.30},
    "claude-haiku-4-5-20251001":    {"input":  1.00, "output":  5.00, "cache_read": 0.10},
    # ── Legacy 4.x (pricier than current equivalents) ─────────────
    "claude-opus-4-5-20251101":     {"input":  5.00, "output": 25.00, "cache_read": 0.50},
    "claude-opus-4-1-20250805":     {"input": 15.00, "output": 75.00, "cache_read": 1.50},
    "claude-opus-4-20250514":       {"input": 15.00, "output": 75.00, "cache_read": 1.50},
    "claude-sonnet-4-5-20250929":   {"input":  3.00, "output": 15.00, "cache_read": 0.30},
    "claude-sonnet-4-20250514":     {"input":  3.00, "output": 15.00, "cache_read": 0.30},
    # ── Legacy 3.x — cheapest options ────────────────────────────
    "claude-haiku-3-5-20241022":    {"input":  0.80, "output":  4.00, "cache_read": 0.08},
    "claude-haiku-3-20240307":      {"input":  0.25, "output":  1.25, "cache_read": 0.03,
                                     "deprecated": "⚠️ 19 Apr 2026"},
    "claude-sonnet-3-5-20241022":   {"input":  3.00, "output": 15.00, "cache_read": 0.30},
    "claude-sonnet-3-7-20250219":   {"input":  3.00, "output": 15.00, "cache_read": 0.30},
}

# --- Capability helper ---
# Safely check if a capability (e.g. thinking, code_execution) is supported
def supported(caps_obj, attr):
    try:
        return getattr(getattr(caps_obj, attr), "supported", False)
    except AttributeError:
        return False

# --- Build the comparison table ---
# batch input price = 50% of standard input price
rows = []
for m in models:
    c = m.capabilities
    p = PRICING.get(m.id, {})
    rows.append({
        "model_id":      m.id,
        "input $/MTok":  p.get("input",      "?"),
        "output $/MTok": p.get("output",     "?"),
        "cache_read":    p.get("cache_read", "?"),
        "batch input":   round(p["input"] * 0.5, 3) if p.get("input") else "?",
        "thinking":      "✓" if supported(c, "thinking")       else "·",
        "code_exec":     "✓" if supported(c, "code_execution")  else "·",
        "status":        p.get("deprecated", "ok"),
    })

# Convert to DataFrame, coerce price columns to numeric, and sort by input cost
df = pd.DataFrame(rows)
df["input $/MTok"]  = pd.to_numeric(df["input $/MTok"],  errors="coerce")
df["output $/MTok"] = pd.to_numeric(df["output $/MTok"], errors="coerce")
df["cache_read"]    = pd.to_numeric(df["cache_read"],    errors="coerce")
df["batch input"]   = pd.to_numeric(df["batch input"],   errors="coerce")
df = df.sort_values("input $/MTok", na_position="last")
display(df.to_string(index=False))

# --- Cost estimator ---
# Example workload: 100 calls x 2 000 input tokens + 500 output tokens
print("── Cost per practice session ───────────────────────────────────────────")
for mid, p in sorted(PRICING.items(), key=lambda x: x[1].get("input", 999)):
    if "input" not in p:
        continue
    cost = (2000 * 100 / 1e6) * p["input"] + (500 * 100 / 1e6) * p["output"]
    dep  = f"  {p.get("deprecated","")}" if p.get("deprecated") else ""
    print(f"  {mid:<42}  ${cost:.4f}{dep}")


'                  model_id  input $/MTok  output $/MTok  cache_read  batch input thinking code_exec status\n claude-haiku-4-5-20251001           1.0            5.0         0.1          0.5        ✓         ·     ok\n         claude-sonnet-4-6           3.0           15.0         0.3          1.5        ✓         ✓     ok\n  claude-sonnet-4-20250514           3.0           15.0         0.3          1.5        ✓         ·     ok\nclaude-sonnet-4-5-20250929           3.0           15.0         0.3          1.5        ✓         ✓     ok\n  claude-opus-4-5-20251101           5.0           25.0         0.5          2.5        ✓         ✓     ok\n           claude-opus-4-6           5.0           25.0         0.5          2.5        ✓         ✓     ok\n  claude-opus-4-1-20250805          15.0           75.0         1.5          7.5        ✓         ·     ok\n    claude-opus-4-20250514          15.0           75.0         1.5          7.5        ✓         ·     ok\n   claude-3-haiku-20240307 

'                  model_id  input $/MTok  output $/MTok  cache_read  batch input thinking code_exec status\n claude-haiku-4-5-20251001           1.0            5.0         0.1          0.5        ✓         ·     ok\n         claude-sonnet-4-6           3.0           15.0         0.3          1.5        ✓         ✓     ok\n  claude-sonnet-4-20250514           3.0           15.0         0.3          1.5        ✓         ·     ok\nclaude-sonnet-4-5-20250929           3.0           15.0         0.3          1.5        ✓         ✓     ok\n  claude-opus-4-5-20251101           5.0           25.0         0.5          2.5        ✓         ✓     ok\n           claude-opus-4-6           5.0           25.0         0.5          2.5        ✓         ✓     ok\n  claude-opus-4-1-20250805          15.0           75.0         1.5          7.5        ✓         ·     ok\n    claude-opus-4-20250514          15.0           75.0         1.5          7.5        ✓         ·     ok\n   claude-3-haiku-20240307 


── Coste por sesión de práctica ─────────────────────────────────
  claude-haiku-3-20240307                     $0.1125  ⚠️ 19 Apr 2026
  claude-haiku-3-5-20241022                   $0.3600
  claude-haiku-4-5-20251001                   $0.4500
  claude-sonnet-4-6                           $1.3500
  claude-sonnet-4-5-20250929                  $1.3500
  claude-sonnet-4-20250514                    $1.3500
  claude-sonnet-3-5-20241022                  $1.3500
  claude-sonnet-3-7-20250219                  $1.3500
  claude-opus-4-6                             $2.2500
  claude-opus-4-5-20251101                    $2.2500
  claude-opus-4-1-20250805                    $6.7500
  claude-opus-4-20250514                      $6.7500


In [9]:
# List all model IDs currently returned by the API
for m in models:
    print(m.id)

claude-sonnet-4-6
claude-opus-4-6
claude-opus-4-5-20251101
claude-haiku-4-5-20251001
claude-sonnet-4-5-20250929
claude-opus-4-1-20250805
claude-opus-4-20250514
claude-sonnet-4-20250514
claude-3-haiku-20240307
